In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
LEFT = 0
DOWN = 1
RIGHT = 2
UP = 3
UPLEFT=4
UPRIGHT=5
DOWNLEFT=6
DOWNRIGHT=7

In [ ]:
class RLTables:

    def __init__(self, map_size,goal_state,actions):
        self.map_size = map_size
        self.total_states = map_size * map_size
        self.goal_state = goal_state
        self.actions=actions

    def initialize_value_table(self):

        self.value_table = np.zeros(self.total_states)    

    def initialize_q_table(self):

        self.q_table = np.zeros((self.total_states, self.actions))

    def initialize_random_policy(self):

        self.policy = np.random.randint(0,self.actions,self.total_states)
   
       
        self.policy[self.goal_state] = -1


    def print_all_tables(self):

        print("\nQ-Table")
        print(self.q_table)
        print("Policy")
        print(self.policy)
        print("Reward")
        print(self.reward)

    def get_tables(self):
        return self.policy, self.value_table, self.q_table, self.reward


    def initialize_reward(self):
        self.reward = np.full(self.total_states,-1)
        self.reward[self.goal_state] = 10

    def initialize_all_tables(self):
        self.initialize_random_policy()
        self.initialize_value_table()
        self.initialize_q_table()
        self.initialize_reward()

In [ ]:
class Plot(RLTables):
    def __init__(self, map_size,goal_state,actions,policy,value,q,reward):
        super().__init__(map_size,goal_state=goal_state,actions=actions)
        self.directions = {-1: "G", 0: "\u2190", 1: "\u2193", 2: "\u2192", 3: "\u2191",4:"\u2196",5:"\u2197",6:"\u2199",7:"\u2198",8:"S"}
        self.policy, self.value_table, self.q_table, self.reward=policy,value,q,reward
        
    def convert_policy_to_arrows(self):

        def replace_with_direction(num):
            return self.directions[num]

        vectorized_replace = np.vectorize(replace_with_direction)

        result_array = vectorized_replace(self.policy)

        return result_array

    def show_policy_table(self):
        arrow_based_policy = self.convert_policy_to_arrows()

        self.data = list(arrow_based_policy.reshape(self.map_size, self.map_size))

        # Create a figure and axis
        _, ax = plt.subplots()
        colors = plt.cm.BuPu(np.full((self.map_size, self.map_size), 0.1))


        self.table = ax.table(cellText=self.data, loc='center', cellLoc='center', colLabels=None, cellColours=colors)

        self.table.scale(2.5,3.5)

        # Hide axis
        ax.axis('off')

        # Customize cell properties (optional)
        self.table.auto_set_font_size(False)
        self.table.set_fontsize(25)
        self.table.scale(1.5, 1.5)  # Adjust cell size as needed
        ax.axis('off')  # Turn off the axis

    def show_value_table(self):

        self.show_policy_table()

        if self.map_size == 8:
            self.table.set_fontsize(10)

        for i in range(self.map_size):
                    for j in range(self.map_size):

                            cell_text = self.data[i][j]

                            if cell_text == " " or cell_text == "G":
                                    continue

                            cell = self.table[i, j]

                            number_text = str(round( self.value_table[self.map_size*i + j], 2))
                            full_text = cell_text + '\n' + number_text
                            cell.get_text().set_text(full_text)


    def show_q_table(self):

        self.show_policy_table()

        self.table.set_fontsize(8)


        for i in range(self.map_size):
                        for j in range(self.map_size):

                                cell_text = self.data[i][j]

                                if cell_text == " " or cell_text == "G":
                                        continue


                                cell = self.table[i, j]

                                number_text2 = str(round( self.q_table[self.map_size*i + j][0], 2))
                                number_text4 = str(round( self.q_table[self.map_size*i + j][1], 2))
                                number_text3 = str(round( self.q_table[self.map_size*i + j][2], 2))
                                number_text1 = str(round( self.q_table[self.map_size*i + j][3], 2))
                                number_text5 = str(round( self.q_table[self.map_size*i + j][4], 2))
                                number_text6 = str(round( self.q_table[self.map_size*i + j][5], 2))
                                number_text7 = str(round( self.q_table[self.map_size*i + j][6], 2))
                                number_text8 = str(round( self.q_table[self.map_size*i + j][7], 2))
                                number_text9 = str(round( self.q_table[self.map_size*i + j][8], 2))


                                full_text = number_text1 +"  "+number_text4+"  "+number_text8 + "\n\n" + number_text2 +"\n\n"+  number_text3 +"    " + cell_text + "    " + number_text5 + "\n\n" + number_text6 + "  "+ number_text7+"  "+number_text9

                                cell.get_text().set_text(full_text)


    def show_all_tables(self):
        self.show_policy_table()
        self.show_q_table()


## Kings Moves

In [ ]:
class GridWorldKingsMoves(RLTables):
    def __init__(self,map_size,goal_state,epsilon,alpha,gamma,start_state):
        super().__init__(map_size=map_size,goal_state=goal_state,actions=9)

        self.initialize_all_tables()
        
        self.epsilon=epsilon
        self.alpha=alpha
        self.gamma=gamma


        self.wind_strength = [0, 0, 0, 1, 1, 1, 2, 2, 1, 0]
          
        self.current_state=0
        self.start_state=start_state

    def move(self,row,col,action):
        if action == 0:
            col=max(col-1,0)
        elif action == 1:
            row=min(row+1,self.map_size-1)
        elif action == 2:
            col=min(self.map_size-1,col+1)
        elif action == 3:
            row=max(row-1,0)
        elif action == 4:  
            row = max(row - 1, 0)
            col = max(col - 1, 0)
        elif action == 5:  
            row = max(row - 1, 0)
            col = min(col + 1, self.map_size - 1)
        elif action == 6:  
            row = min(row + 1, self.map_size - 1)
            col = max(col - 1, 0)
        elif action == 7:  
            row = min(row + 1, self.map_size - 1)
            col = min(col + 1, self.map_size - 1)
        elif action == 8:
            row=row
            col=col
            
    
        next_state=row*self.map_size+col
        
        return next_state
    
    def step(self,action):
        row,col=divmod(self.current_state, self.map_size)
        next_state=self.move(row,col,action)
        wind_dir=self.wind_strength[col]
        
        # due to wind strength agent pushed in wind direction i.e upper rows
        row,col=divmod(next_state, self.map_size)
        next_state=max(row-wind_dir,0)*self.map_size+col

        # get the reward
        next_reward=self.reward[next_state]

        terminated=False
        if next_state==self.goal_state:
            terminated=True

        return next_state,next_reward,terminated
    
    def reset(self):
        self.current_state=self.start_state
        return self.current_state,-1,False
    
    def trajectory(self):
        trajectory=[self.start_state]
        terminated=False
        self.current_state=self.start_state
        while not terminated:
            action=self.policy[self.current_state]
            next_state,_,terminated=self.step(action)
            trajectory.append(next_state)
            self.current_state=next_state

        return trajectory
        
    def epsilon_greedy_action(self,s):
        random=np.random.rand()
        if random<self.epsilon:
            action=np.random.randint(0,self.actions)
        else:
            action=np.argmax(self.q_table[s])
            
        return action

        
    def SARSA(self,max_episodes):
        for _ in range(max_episodes):
            
            self.current_state,_,terminated=self.reset()
            
            action=self.epsilon_greedy_action(self.current_state)
            
            while terminated is not True:
                    
                next_state,reward,terminated=self.step(action)

                if terminated:
                    target=reward
                else:
                    next_action=self.epsilon_greedy_action(next_state)
                    target=reward+self.gamma*self.q_table[next_state,next_action]
                        
                # SARSA update step
                self.q_table[self.current_state,action]+=self.alpha*(target-self.q_table[self.current_state,action])
                self.current_state=next_state
                action=next_action
           
            self.epsilon = max(self.epsilon * 0.99995, 0.05)
            #self.alpha = max(self.alpha * 0.95, 0.005)

          
        for s in range(self.total_states):  
            if s!=self.goal_state:
                self.policy[s]=np.argmax(self.q_table[s])

    
    def Q_learning(self,max_episodes):
        for _ in range(max_episodes):

            self.current_state,_,terminated=self.reset()
            
            while not terminated:
                action=self.epsilon_greedy_action(self.current_state)

                next_state,reward,terminated=self.step(action)

                if terminated:
                    target=reward
                else:
                    target=reward+self.gamma*self.q_table[next_state].max()

               
                self.q_table[self.current_state,action]+=self.alpha*(target-self.q_table[self.current_state,action])

                self.current_state=next_state
            
            self.epsilon = max(self.epsilon * 0.99995, 0.05)
           
        for s in range(self.total_states):  
            if s!=self.goal_state:
                self.policy[s]=np.argmax(self.q_table[s])

    def Expected_SARSA(self,max_episodes):
        for _ in range(max_episodes):

            self.current_state,_,terminated=self.reset()
            
            action=self.epsilon_greedy_action(self.current_state)
            
            while terminated is not True:
                    
                next_state,reward,terminated=self.step(action)

                if terminated:
                    target=reward
                else:
                    greedy_action=np.argmax(self.q_table[next_state])
                    expected_q=0
                    for a_ in range(self.actions):
                        if a_ == greedy_action:
                            prob=1-self.epsilon+(self.epsilon/self.actions)
                        else:
                            prob=(self.epsilon/self.actions)

                        expected_q+=prob*self.q_table[next_state,a_]

                    target=reward+self.gamma*expected_q
                        
                # SARSA update step
                self.q_table[self.current_state,action]+=self.alpha*(target-self.q_table[self.current_state,action])
                self.current_state=next_state
                action=self.epsilon_greedy_action(self.current_state)
            
            self.epsilon = max(self.epsilon * 0.99995, 0.05)
            
        
        for s in range(self.total_states):  
            if s!=self.goal_state:
                self.policy[s]=np.argmax(self.q_table[s])

    def MonteCarlo(self,max_episodes):

        C=np.zeros_like(self.q_table)
    
        for _ in range(max_episodes): 

            self.current_state,_,terminated=self.reset()
            episode=[]
        
            while not terminated: 
                action = self.epsilon_greedy_action(self.current_state)
                next_state,reward,terminated=self.step(action)

                episode.append((self.current_state,action,reward))
                self.current_state=next_state

            Returns=0
            W=1

            
            for state,action,reward in reversed(episode):
                Returns=self.gamma*Returns+reward

                C[state,action]+=W

                self.q_table[state,action]+=(W/C[state,action])*(Returns-self.q_table[state,action])

                self.policy[state]=np.argmax(self.q_table[state])

                

                if action==self.policy[state]:
                    W/=(1-self.epsilon+self.epsilon/self.actions)
                else:
                    W=0

                if W==0:
                    break

                
            self.epsilon = max(self.epsilon * 0.99995, 0.05)
           

### SARSA

In [ ]:
env=GridWorldKingsMoves(10,37,0.65,1.0/9.0,1,30)
env.SARSA(5000)
trajectory=env.trajectory()
print(f"Policy Found: {env.policy}")
print(f"Trajectory : {trajectory}")

In [ ]:
policy,value,q,reward=env.get_tables()
plot = Plot(map_size=10,goal_state=37,actions=9,policy=policy,value=value,q=q,reward=reward)
plot.show_all_tables()

### Q-learning

In [ ]:
env=GridWorldKingsMoves(10,37,0.85,1.0/9.0,1,30)
env.Q_learning(5000)
trajectory=env.trajectory()
print(f"Policy Found: {env.policy}")
print(f"Trajectory : {trajectory}")

In [ ]:
policy,value,q,reward=env.get_tables()
plot = Plot(map_size=10,goal_state=37,actions=9,policy=policy,value=value,q=q,reward=reward)
plot.show_all_tables()

### Expected SARSA

In [ ]:
env=GridWorldKingsMoves(10,37,0.65,1.0/9.0,1,30)
env.Expected_SARSA(5000)
trajectory=env.trajectory()
print(f"Policy Found: {env.policy}")
print(f"Trajectory : {trajectory}")

In [ ]:
policy,value,q,reward=env.get_tables()
plot = Plot(map_size=10,goal_state=37,actions=9,policy=policy,value=value,q=q,reward=reward)
plot.show_all_tables()

### MONTE CARLO

In [ ]:
env=GridWorldKingsMoves(10,37,0.85,1.0/9.0,1,30)
env.MonteCarlo(5000)
trajectory=env.trajectory()
print(f"Policy Found: {env.policy}")
print(f"Trajectory : {trajectory}")

In [ ]:
policy,value,q,reward=env.get_tables()
plot = Plot(map_size=10,goal_state=37,actions=9,policy=policy,value=value,q=q,reward=reward)
plot.show_all_tables()

## Stochastic Wind

In [ ]:
class GridWorldStochastic(RLTables):
    def __init__(self,map_size,goal_state,epsilon,alpha,gamma,start_state):
        super().__init__(map_size=map_size,goal_state=goal_state,actions=9)

        self.initialize_all_tables()
        
        self.epsilon=epsilon
        self.alpha=alpha
        self.gamma=gamma


        self.wind_strength = [0, 0, 0, 1, 1, 1, 2, 2, 1, 0]
          
        self.current_state=0
        self.start_state=start_state

    def move(self,row,col,action):
        if action == 0:
            col=max(col-1,0)
        elif action == 1:
            row=min(row+1,self.map_size-1)
        elif action == 2:
            col=min(self.map_size-1,col+1)
        elif action == 3:
            row=max(row-1,0)
        elif action == 4:  
            row = max(row - 1, 0)
            col = max(col - 1, 0)
        elif action == 5:  
            row = max(row - 1, 0)
            col = min(col + 1, self.map_size - 1)
        elif action == 6:  
            row = min(row + 1, self.map_size - 1)
            col = max(col - 1, 0)
        elif action == 7:  
            row = min(row + 1, self.map_size - 1)
            col = min(col + 1, self.map_size - 1)
        elif action == 8:
            row=row
            col=col
            
    
        next_state=row*self.map_size+col
        
        return next_state
    
    def step(self,action):
        row,col=divmod(self.current_state, self.map_size)
        next_state=self.move(row,col,action)
        wind_dir=self.wind_strength[col]

        wind_effect=np.random.choice(
            [max(0, wind_dir - 1), wind_dir, wind_dir + 1]
        )
        
        # due to wind strength agent pushed in wind direction i.e upper rows
        row,col=divmod(next_state, self.map_size)
        next_state=max(row-wind_effect,0)*self.map_size+col

        # get the reward
        next_reward=self.reward[next_state]

        terminated=False
        if next_state==self.goal_state:
            terminated=True

        return next_state,next_reward,terminated
    
    def reset(self):
        self.current_state=self.start_state
        return self.current_state,-1,False
    
    def trajectory(self):
        trajectory=[self.start_state]
        terminated=False
        self.current_state=self.start_state
        while not terminated:
            action=self.policy[self.current_state]
            next_state,_,terminated=self.step(action)
            trajectory.append(next_state)
            self.current_state=next_state

        return trajectory
        
    def epsilon_greedy_action(self,s):
        random=np.random.rand()
        if random<self.epsilon:
            action=np.random.randint(0,self.actions)
        else:
            action=np.argmax(self.q_table[s])
            
        return action

        
    def SARSA(self,max_episodes):
        for _ in range(max_episodes):
            
            self.current_state,_,terminated=self.reset()
            
            action=self.epsilon_greedy_action(self.current_state)
            
            while terminated is not True:
                    
                next_state,reward,terminated=self.step(action)

                if terminated:
                    target=reward
                else:
                    next_action=self.epsilon_greedy_action(next_state)
                    target=reward+self.gamma*self.q_table[next_state,next_action]
                        
                # SARSA update step
                self.q_table[self.current_state,action]+=self.alpha*(target-self.q_table[self.current_state,action])
                self.current_state=next_state
                action=next_action
           
            self.epsilon = max(self.epsilon * 0.99995, 0.05)
          
        for s in range(self.total_states):  
            if s!=self.goal_state:
                self.policy[s]=np.argmax(self.q_table[s])

    
    def Q_learning(self,max_episodes):
        for _ in range(max_episodes):

            self.current_state,_,terminated=self.reset()
            
            while not terminated:
                action=self.epsilon_greedy_action(self.current_state)

                next_state,reward,terminated=self.step(action)

                if terminated:
                    target=reward
                else:
                    target=reward+self.gamma*self.q_table[next_state].max()

               
                self.q_table[self.current_state,action]+=self.alpha*(target-self.q_table[self.current_state,action])

                self.current_state=next_state
            
            self.epsilon = max(self.epsilon * 0.99995, 0.05)
           
        for s in range(self.total_states):  
            if s!=self.goal_state:
                self.policy[s]=np.argmax(self.q_table[s])

    def Expected_SARSA(self,max_episodes):
        for _ in range(max_episodes):

            self.current_state,_,terminated=self.reset()
            
            action=self.epsilon_greedy_action(self.current_state)
            
            while terminated is not True:
                    
                next_state,reward,terminated=self.step(action)

                if terminated:
                    target=reward
                else:
                    greedy_action=np.argmax(self.q_table[next_state])
                    expected_q=0
                    for a_ in range(self.actions):
                        if a_ == greedy_action:
                            prob=1-self.epsilon+(self.epsilon/self.actions)
                        else:
                            prob=(self.epsilon/self.actions)

                        expected_q+=prob*self.q_table[next_state,a_]

                    target=reward+self.gamma*expected_q
                        
                # SARSA update step
                self.q_table[self.current_state,action]+=self.alpha*(target-self.q_table[self.current_state,action])
                self.current_state=next_state
                action=self.epsilon_greedy_action(self.current_state)
            
            self.epsilon = max(self.epsilon * 0.99995, 0.05)
            
        
        for s in range(self.total_states):  
            if s!=self.goal_state:
                self.policy[s]=np.argmax(self.q_table[s])

    def MonteCarlo(self,max_episodes):

        C=np.zeros_like(self.q_table)
    
        for _ in range(max_episodes): 

            self.current_state,_,terminated=self.reset()
            episode=[]
        
            while not terminated: 
                action = self.epsilon_greedy_action(self.current_state)
                next_state,reward,terminated=self.step(action)

                episode.append((self.current_state,action,reward))
                self.current_state=next_state

            Returns=0
            W=1

            
            for state,action,reward in reversed(episode):
                Returns=self.gamma*Returns+reward

                C[state,action]+=W

                self.q_table[state,action]+=(W/C[state,action])*(Returns-self.q_table[state,action])

                self.policy[state]=np.argmax(self.q_table[state])

                

                if action==self.policy[state]:
                    W/=(1-self.epsilon+self.epsilon/self.actions)
                else:
                    W=0

                if W==0:
                    break

                
            self.epsilon = max(self.epsilon * 0.99995, 0.05)
           

### SARSA

In [ ]:
env=GridWorldStochastic(10,37,0.85,1.0/9.0,1,30)
env.SARSA(5000)
trajectory=env.trajectory()
print(f"Policy Found: {env.policy}")
print(f"Trajectory : {trajectory}")

In [ ]:
policy,value,q,reward=env.get_tables()
plot = Plot(map_size=10,goal_state=37,actions=9,policy=policy,value=value,q=q,reward=reward)
plot.show_all_tables()

### Q-learning

In [ ]:
env=GridWorldKingsMoves(10,37,0.1,1.0/9.0,1,30)
env.Q_learning(5000)
trajectory=env.trajectory()
print(f"Policy Found: {env.policy}")
print(f"Trajectory : {trajectory}")

In [ ]:
policy,value,q,reward=env.get_tables()
plot = Plot(map_size=10,goal_state=37,actions=9,policy=policy,value=value,q=q,reward=reward)
plot.show_all_tables()

### Expected SARSA

In [ ]:
env=GridWorldKingsMoves(10,37,0.65,1.0/9.0,1,30)
env.Expected_SARSA(5000)
trajectory=env.trajectory()
print(f"Policy Found: {env.policy}")
print(f"Trajectory : {trajectory}")

In [ ]:
policy,value,q,reward=env.get_tables()
plot = Plot(map_size=10,goal_state=37,actions=9,policy=policy,value=value,q=q,reward=reward)
plot.show_all_tables()

### MONTE CARLO

In [ ]:
env=GridWorldKingsMoves(10,37,0.8,1.0/9.0,1,30)
env.MonteCarlo(5000)
trajectory=env.trajectory()
print(f"Policy Found: {env.policy}")
print(f"Trajectory : {trajectory}")

In [ ]:
policy,value,q,reward=env.get_tables()
plot = Plot(map_size=10,goal_state=37,actions=9,policy=policy,value=value,q=q,reward=reward)
plot.show_all_tables()

## Results and Analysis

- The start state of the grid is 30 and goal is to reach 37th grid (0-indexed). All Algorithms were applied with initial ***epsilon*** equal to ***0.85*** and initial ***alpha*** equal to ***1/N(A)***. Epsilon was gradually decreased episode wise with **epsilon x 0.99995** until it reached the minimum value *0.05*. The initial policy was initialized randomly.

1. **Q-learning** in **Stochastic environment** is converging fastest with mean time of *4-5 secs*.
2. **Monte Carlo** produces most stable policy as it is giving the same optimal policy in *stochastic* as well as in simple *static* environment. Moreover, even tuning initial state of hyperparameters i.e alpha and epsilon , the model produced the same policy after running different experiments.
3. **Expected SARSA** has higher bias as compared to **SARSA** as it uses expected value, which depends on estimated Q and policy, not on the actual sample. On the other hand, it has Lower variance because averaging over all possible actions smooths out randomness.
4. **Q-learning** updates toward the optimal greedy action, even when it actually takes exploratory or suboptimal actions due to this it learns that the shortest path along the cliff yeilds the highest reward in the long run. Due to exploration the actions may cause it to fall off the cliff but it updates towards the greedy optimal action path hugging the cliff.
5. Tuning the initial guess for epsilon (0.1-0.3) all algorithms converged much faster than anticipated except for Monte Carlo which took too long to hault (had to interupt code cell). As the alpha was adapted episode wise the algorithms began to converge slower. Another insight was the when epsilon was kept lower i.e 0.1-0.3 all algorithms except for Monte Carlo gave the same policy when run multiple times. (same policy with respect to itself)

